In [22]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np



In [23]:
# Load the Kaggle Stroke Dataset
data_path = "healthcare-dataset-stroke-data.csv"
df = pd.read_csv(data_path)

# Display the first 5 rows and dataset summary
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
df.head()


Dataset Shape: (5110, 12)

First 5 rows of the dataset:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [24]:
# Display dataset summary
print("\nDataset Summary:")
print(df.info())


Dataset Summary:
<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB
None


In [25]:
# Drop the 'id' column as it carries no predictive value
df_clean = df.drop(columns=['id'])

# Drop initial missing values to obtain a fully intact baseline dataset
df_clean = df_clean.dropna().reset_index(drop=True)

# Encode categorical features into numerical representations
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le

# Display cleaned dataset information
print("Clean Dataset Shape:", df_clean.shape)
print("\nData Types and Missing Values Count:")
print(df_clean.isnull().sum())
print("\nFirst 5 rows of Clean Dataset:")
df_clean.head()

Clean Dataset Shape: (4909, 11)

Data Types and Missing Values Count:
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

First 5 rows of Clean Dataset:


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1,67.0,0,1,1,2,1,228.69,36.6,1,1
1,1,80.0,0,1,1,2,0,105.92,32.5,2,1
2,0,49.0,0,0,1,2,1,171.23,34.4,3,1
3,0,79.0,1,0,1,3,0,174.12,24.0,2,1
4,1,81.0,0,0,1,2,1,186.21,29.0,1,1


In [26]:
# Separate features (X) and target label (y)
X = df_clean.drop(columns=["stroke"])
y = df_clean["stroke"]

# Split dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train the Baseline Random Forest Classifier
# Handle class imbalance using class_weight='balanced'
baseline_rf = RandomForestClassifier(class_weight="balanced", random_state=42)
baseline_rf.fit(X_train, y_train)

# Predict on the test set and calculate metrics
y_pred_baseline = baseline_rf.predict(X_test)

f1_baseline = f1_score(y_test, y_pred_baseline)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
precision_baseline = precision_score(y_test, y_pred_baseline)
recall_baseline = recall_score(y_test, y_pred_baseline)

print("--- Baseline Model Performance (Clean Data) ---")
print(f"Accuracy : {accuracy_baseline:.4f}")
print(f"Precision: {precision_baseline:.4f}")
print(f"Recall   : {recall_baseline:.4f}")
print(f"F1 Score : {f1_baseline:.4f}")

--- Baseline Model Performance (Clean Data) ---
Accuracy : 0.9379
Precision: 0.0870
Recall   : 0.0476
F1 Score : 0.0615


In [28]:
def inject_missing_data(X_data, missing_rate, random_state=42):
    """Randomly inject missing values (NaN) into the feature matrix at a specified rate (MCAR)."""
    np.random.seed(random_state)
    X_missing = X_data.copy().astype(float)

    # Calculate total number of cells and target number of missing values
    n_rows, n_cols = X_missing.shape
    total_cells = n_rows * n_cols
    n_missing = int(total_cells * missing_rate)

    # Randomly choose indices to mask as NaN
    flat_indices = np.random.choice(total_cells, size=n_missing, replace=False)
    row_indices, col_indices = np.unravel_index(flat_indices, (n_rows, n_cols))

    # Apply NaN mask
    X_missing.iloc[row_indices, col_indices] = np.nan
    return X_missing, (row_indices, col_indices)


# Inject 10%, 20%, and 30% missingness into the features
X_miss_10, mask_10 = inject_missing_data(X, missing_rate=0.10)
X_miss_20, mask_20 = inject_missing_data(X, missing_rate=0.20)
X_miss_30, mask_30 = inject_missing_data(X, missing_rate=0.30)

print("--- Missingness Injection Summary ---")
print(
    f"10% Missing Dataset: {X_miss_10.isnull().sum().sum()} NaNs created "
    f"({X_miss_10.isnull().sum().sum() / X.size:.1%})"
)
print(
    f"20% Missing Dataset: {X_miss_20.isnull().sum().sum()} NaNs created "
    f"({X_miss_20.isnull().sum().sum() / X.size:.1%})"
)
print(
    f"30% Missing Dataset: {X_miss_30.isnull().sum().sum()} NaNs created "
    f"({X_miss_30.isnull().sum().sum() / X.size:.1%})"
)

--- Missingness Injection Summary ---
10% Missing Dataset: 31830 NaNs created (64.8%)
20% Missing Dataset: 43830 NaNs created (89.3%)
30% Missing Dataset: 47800 NaNs created (97.4%)
